In [ ]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt


--2026-03-20 03:33:23--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  6.19MB/s    in 0.2s    

2026-03-20 03:33:23 (6.19 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [ ]:
with open('input.txt','r',encoding='utf-8') as file:
  text=file.read()

In [ ]:
print('length of the dataset :', len(text))

length of the dataset : 1115394


In [ ]:
chars=sorted(list(set(text)))
vocab_size=len(chars)
print(vocab_size)

65


In [ ]:
soti ={ ch:i for i,ch in enumerate(chars)}
itos ={i:ch for i,ch in enumerate(chars)}
encode= lambda s: [soti[c] for c in s]
decode = lambda s: ''.join([itos[c] for c in s])

print(encode("hi hello"))
print(decode(encode("hi hello")))


[46, 47, 1, 46, 43, 50, 50, 53]
hi hello


In [ ]:
import torch

In [ ]:
data = torch.tensor(encode(text),dtype=torch.long)
print(data.shape,data.type())

torch.Size([1115394]) torch.LongTensor


In [ ]:
n= int(0.9*len(data))
train_data=data[:n]
test_data=data[n:]

In [ ]:

batch_size=4
block_size=8
def get_batch(string):
  if string=='train':
    data=train_data
  else:
    data=test_data
  ix=torch.randint(len(data)-block_size,(batch_size,))
  x=torch.stack([data[i:i+block_size] for i in ix])
  y=torch.stack([data[i+1:i+1+block_size] for i in ix])
  return x,y

xb,yb=get_batch('train')
print('inputs')
print(xb.shape)
print(xb)
print('targets')
print(yb.shape)
print(yb)
print('============================================================================')
for b in range(batch_size):
  for c in range(block_size):
    context=xb[b,:c+1]
    target=yb[b,c]
    print(f'for this context {context.tolist()} the target is {target}')


inputs
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
for this context [24] the target is 43
for this context [24, 43] the target is 58
for this context [24, 43, 58] the target is 5
for this context [24, 43, 58, 5] the target is 57
for this context [24, 43, 58, 5, 57] the target is 1
for this context [24, 43, 58, 5, 57, 1] the target is 46
for this context [24, 43, 58, 5, 57, 1, 46] the target is 43
for this context [24, 43, 58, 5, 57, 1, 46, 43] the target is 39
for this context [44] the target is 53
for this context [44, 53] the target is 56
for this context [44, 53, 56] the target is 1
for this context [44, 53, 56, 1] the target is 58
for this context 

the FINAL SCRIPT



In [ ]:
#final script to run

import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

#hyperparameters
n_embed=64
batch_size=16
block_size=32
max_iter=8000
eval_iter=200
eval_interval=500
learning_rate=1e-3
dropout=0.2
n_layer=4
n_head=4

#data import
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt  # remove ! for script
with open('input.txt','r',encoding='utf-8') as file:
  text=file.read()
chars=sorted(list(set(text)))
vocab_size=len(chars)

#encoder and decoder
soti ={ ch:i for i,ch in enumerate(chars)}
itos ={i:ch for i,ch in enumerate(chars)}
encode= lambda s: [soti[c] for c in s]
decode = lambda s: ''.join([itos[c] for c in s])

#train data and test data split
data = torch.tensor(encode(text),dtype=torch.long)
n= int(0.9*len(data))
train_data=data[:n]
test_data=data[n:]

def get_batch(string):
  if string=='train':
    data=train_data
  else:
    data=test_data
  ix=torch.randint(len(data)-block_size,(batch_size,))
  x=torch.stack([data[i:i+block_size] for i in ix])
  y=torch.stack([data[i+1:i+1+block_size] for i in ix])
  return x,y

@torch.no_grad()
def estimate_loss():
  out={}
  model.eval()
  for split in ['train', 'val']:
    losses = torch.zeros(eval_iter)
    for k in range(eval_iter):
      x,y=get_batch(split)
      logit,loss=model(x,y)

      losses[k]=loss.item()
    out[split]=losses.mean()
  model.train()
  return out

class head(nn.Module):
  def __init__(self,head_size):
    super().__init__()
    self.query=nn.Linear(n_embed,head_size,bias=False)
    self.key=nn.Linear(n_embed,head_size,bias=False)
    self.value=nn.Linear(n_embed,head_size,bias=False)
    self.register_buffer('tril',torch.tril(torch.ones(block_size,block_size)))
    self.dropout = nn.Dropout(dropout)

  def forward(self,x):
    B,T,C = x.shape
    k = self.key(x)   # (B,T,C)
    q = self.query(x) # (B,T,C)
    # compute attention scores ("affinities")
    wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
    wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
    wei = F.softmax(wei, dim=-1) # (B, T, T)
    wei = self.dropout(wei)
    # perform the weighted aggregation of the values
    v = self.value(x) # (B,T,C)
    out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
    return out

class MultiHeadAttention(nn.Module):
  # mulptile heads running in parallel instead of one
  def __init__(self,num_heads,head_size):
    super().__init__()
    self.heads = nn.ModuleList([head(head_size) for _ in range(num_heads)])
    self.proj=nn.Linear(n_embed,n_embed)
    self.dropout = nn.Dropout(dropout)

  def forward(self,x):
    out=torch.cat([h(x) for h in self.heads],dim=-1)
    out = self.dropout(self.proj(out))
    return out

class FeedForward(nn.Module):
  #simple ffedforward network with linaer followed by non linear function
  def __init__(self,n_embed):
    super().__init__()
    self.net=nn.Sequential(
        nn.Linear(n_embed,4*n_embed),
        nn.ReLU(),
        nn.Linear(4*n_embed,n_embed), # projection layer
        nn.Dropout(dropout),
    )

  def forward(self,x):
    return self.net(x)

class block(nn.Module):
  # this is to create a block where attention and CNN are together , so running this will do the attention process and FFD
  def __init__(self,n_embed,num_head):
    super().__init__()
    self.head_size=n_embed//num_head
    self.LM_head=MultiHeadAttention(num_head,self.head_size)
    self.FFD=FeedForward(n_embed)
    self.ln1=nn.LayerNorm(n_embed)
    self.ln2=nn.LayerNorm(n_embed)

  def forward(self,x):
    x=x+self.LM_head(self.ln1(x))
    x=x+self.FFD(self.ln2(x))
    return x

class biagramLanguageModel(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.token_embeding_table=nn.Embedding(vocab_size,n_embed)
    self.pos_embedding_table=nn.Embedding(block_size,n_embed)
    self.lm__head=nn.Linear(n_embed,vocab_size)
    self.blocks = nn.Sequential(*[block(n_embed, n_head) for _ in range(n_layer)])

    self.ln_f = nn.LayerNorm(n_embed) # final layer norm
    self.lm_head = nn.Linear(n_embed, vocab_size)


  def forward(self,input,targets=None):
    B,T=input.shape
    token_emb=self.token_embeding_table(input)  # returns of shape (B,T,n_embed)
    pos_emb=self.pos_embedding_table(torch.arange(T)) #returns (T,n_embed)
    x=token_emb+pos_emb
    x=self.blocks(x)
    x=self.ln_f(x)
    logits=self.lm__head(x)

    if targets==None:
      loss=None
    else:
      b,t,c=logits.shape
      logits=logits.view(b*t,c)
      targets=targets.view(b*t)
      loss=F.cross_entropy(logits,targets)

    return logits,loss

  def generate(self, idx, max_new_tokens):
    for i in range(max_new_tokens):
      idx_cond=idx[:,-block_size:]# to not exceed block size
      logits,loss= self(idx_cond) #returns logits in shape
      logits=logits[:,-1,:] #returns only (b,c)
      prob=F.softmax(logits,dim=1)
      next_idx=torch.multinomial(prob,num_samples=1)# produce only next xharaxter from the prob
      idx=torch.concat((idx,next_idx),dim=1)
    return idx

model=biagramLanguageModel(vocab_size)
m=model
optimizer=torch.optim.AdamW(m.parameters(),lr=1e-3)

for iter in range(max_iter):
  if iter % eval_interval == 0:
    losses=estimate_loss()
    print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
  xb,yb=get_batch('train')

  logits,loss=model(xb,yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()

print(loss.item())


idx=torch.zeros((1,1),dtype=torch.long)
print(decode(model.generate(idx,2000)[0].tolist()))

--2026-04-05 05:29:16--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.7’

input.txt.7         100%[===================>]   1.06M  --.-KB/s    in 0.005s  

2026-04-05 05:29:16 (228 MB/s) - ‘input.txt.7’ saved [1115394/1115394]

step 0: train loss 4.3439, val loss 4.3318
step 500: train loss 2.3574, val loss 2.3586
step 1000: train loss 2.1786, val loss 2.2035
step 1500: train loss 2.0881, val loss 2.1299
step 2000: train loss 2.0081, val loss 2.0509
step 2500: train loss 1.9553, val loss 2.0085
step 3000: train loss 1.9006, val loss 1.9869
step 3500: train loss 1.8594, val loss 1.9539
step 4000: train loss 1.8313, val 

In [ ]:
print(decode(model.generate(idx,2000)[0].tolist()))


As traw thy there neet! Thy lorss,
Bot happeny kindery bring; soff.

QUEEN VINCENTRETH:
Coumpan me cinne a grace romberalf.
What; when nighty, thabpe frow herebume of theself the our grown confory tLurbit the fraged.
Some a heared, hamet to them I shall
was in us hmovester for will them core,
And be death reeending her ands hith dage: the stray boaughts! threfond Mendernest.
Thou, for slay thin yours usurty remtuder it too misnly bentles be's Greeng;
Gu was say my here'seling havose,
Was speak you own the faades,
Compoitin anse thee, his weaw deatwirs.

JULIEd:
That sin the heads let'stry, hath with this didning day,
Dider and My prighter. O triety grat Shall the helprown the you,
PatiMAgute ow, all appy I beath not her?
Wesell kmack yet, thy durrange wouar
to to No pargy like ap tarl u to your pidisant?
Led For her how to hange Apace bueeguass.

LADY GAUEN ELIZABE:
Heoved suse good, be a mother I am firm,.

Come,
Ourphate housen, thou more. Bastate I imour new,
Wor. Be
FiDUKE VINNY y